In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════

DELTA_PATH    = "/Volumes/workspace/default/network_data/delta/"
FEATURES_PATH = "/Volumes/workspace/default/network_data/features_delta/"
WINDOW_SECONDS   = 1
GAP_THRESHOLD_S  = 5   # segundos — gap mayor que esto activa is_after_gap

# Puertos dominantes normales (corresponden a los 5 SCADA tags)
NORMAL_SRC_PORTS = {"54592", "53260", "53250", "52544", "53326", "53312", "53508"}
SCADA_TAGS       = ["HMI_FIT201", "HMI_LIT101", "HMI_LIT401", "HMI_LIT301", "HMI_AIT202"]

# appl_names anómalos en entorno SCADA industrial
ANOMALOUS_APPL = ["VNC", "Remote Desktop Protocol", "Web Browsing",
                  "Server Message Block (SMB)", "Kaspersky Lab-update",
                  "Google Chrome", "SSDP", "Simple Object Access Protocol"]

# ══════════════════════════════════════════════════════════════════════════════
# 1. LECTURA DEL DELTA
# ══════════════════════════════════════════════════════════════════════════════

df = spark.read.format("delta").load(DELTA_PATH)

# ══════════════════════════════════════════════════════════════════════════════
# 2. SANEAMIENTO ROBUSTO DE COLUMNAS DE ENTRADA
# ══════════════════════════════════════════════════════════════════════════════

# proto: marcar valores no estándar como corruptos
# Modbus_Function_Code: puede contener IPs u otros valores no numéricos
# service (puerto destino): puede contener strings no numéricos
# s_port: puede contener valores no numéricos
# Modbus_Function_Description: normalizar typo "Responqe" → tratar como Response
# Modbus_Transaction_ID: numérico
# Payload: contar bytes hex en Modbus_Value
# window_id: bucket temporal

# Constante para no repetir el string

NON_MODBUS = "Non-Modbus"

df_clean = df \
    .withColumn("proto_clean",
        F.when(F.col("proto").isin("tcp", "udp"), F.col("proto")).otherwise(None)
    ) \
    .withColumn("has_corrupt_proto",
        F.when(F.col("proto").isNotNull() & F.col("proto_clean").isNull(), 1).otherwise(0)
    ) \
    .withColumn("func_code_clean",
        F.when(F.col("Modbus_Function_Code") == NON_MODBUS, None)
         .otherwise(F.expr("try_cast(Modbus_Function_Code as INT)"))
    ) \
    .withColumn("has_corrupt_function_code",
        F.when(
            F.col("Modbus_Function_Code").isNull() |
            (F.col("Modbus_Function_Code") == NON_MODBUS), 0
        ).when(F.col("func_code_clean").isNull(), 1)
         .otherwise(0)
    ) \
    .withColumn("service_clean",
        F.expr("try_cast(service as INT)")
    ) \
    .withColumn("has_corrupt_dst_port",
        F.when(F.col("service").isNotNull() & F.col("service_clean").isNull(), 1).otherwise(0)
    ) \
    .withColumn("s_port_clean",
        F.expr("try_cast(s_port as INT)").cast(StringType())
    ) \
    .withColumn("desc_clean",
        F.when(F.col("Modbus_Function_Description") == NON_MODBUS, None)
         .when(
            F.col("Modbus_Function_Description").rlike("(?i)respon"),
            F.regexp_replace("Modbus_Function_Description", "(?i)respon[a-z]*", "Response")
         ).otherwise(F.col("Modbus_Function_Description"))
    ) \
    .withColumn("tx_id_numeric",
        F.when(F.col("Modbus_Transaction_ID") == NON_MODBUS, None)
         .otherwise(F.expr("try_cast(Modbus_Transaction_ID as DOUBLE)"))
    ) \
    .withColumn("payload_bytes",
        F.when(
            F.col("Modbus_Value").contains("0x"),
            F.size(F.split(F.col("Modbus_Value"), "0x")) - 1
        ).otherwise(0)
    ) \
    .withColumn("window_id",
        (F.unix_timestamp("timestamp") / WINDOW_SECONDS).cast("long")
    )



In [0]:
display(df_clean.limit(10))

In [0]:
display(df_clean.describe())

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. CONNECTION DURATION — self join request/response por transaction_id
# ══════════════════════════════════════════════════════════════════════════════

df_req = df_clean \
    .filter(~F.col("desc_clean").rlike("(?i)response") &
             F.col("desc_clean").isNotNull()) \
    .select(
        F.col("Modbus_Transaction_ID").alias("tx_id"),
        F.col("window_id").alias("req_window"),
        F.unix_timestamp("timestamp").cast("double").alias("req_ts")
    )

df_resp = df_clean \
    .filter(F.col("desc_clean").rlike("(?i)response")) \
    .select(
        F.col("Modbus_Transaction_ID").alias("tx_id"),
        F.unix_timestamp("timestamp").cast("double").alias("resp_ts")
    )

df_durations = df_req.join(df_resp, on="tx_id", how="inner") \
    .withColumn("duration_ms", (F.col("resp_ts") - F.col("req_ts")) * 1000) \
    .filter(F.col("duration_ms") >= 0) \
    .groupBy("req_window").agg(
        F.mean("duration_ms").alias("avg_connection_duration_ms"),
        F.max("duration_ms").alias("max_connection_duration_ms"),
        F.min("duration_ms").alias("min_connection_duration_ms")
    ).withColumnRenamed("req_window", "window_id")



In [0]:
display(df_req.limit(10))
display(df_resp.limit(10))
display(df_durations.limit(10))

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. FEATURE EXTRACTION POR VENTANA
# ══════════════════════════════════════════════════════════════════════════════

features = df_clean.groupBy("window_id").agg(

    # ── Metadatos temporales de la ventana
    F.min("timestamp").alias("window_start"),
    F.max("timestamp").alias("window_end"),

    # ── GRUPO 1: Volumen
    F.count("*").alias("total_packets"),
    F.count(F.when(F.col("i_f_dir") == "outbound", 1)).alias("outbound_packets"),
    F.count(F.when(F.col("i_f_dir") == "inbound",  1)).alias("inbound_packets"),

    # ── GRUPO 2: Protocolo
    F.count(F.when(F.col("proto_clean") == "udp", 1)).alias("udp_packets"),
    F.count(F.when(F.col("proto_clean") == "tcp", 1)).alias("tcp_packets"),
    F.max(F.col("has_corrupt_proto")).alias("has_corrupt_proto"),

    # ── GRUPO 3: appl_name — CIP normal y anómalos
    F.count(F.when(F.col("appl_name") == "CIP_read_tag_service", 1)).alias("cip_read_tag_packets"),
    F.count(F.when(F.col("appl_name").isin(ANOMALOUS_APPL), 1)).alias("anomalous_appl_packets"),
    *[F.max(F.when(F.col("appl_name") == appl, 1).otherwise(0)).alias(
        f"has_{appl.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')}"
      ) for appl in ANOMALOUS_APPL],

    # ── GRUPO 4: Function Codes
    F.count(F.when(F.col("func_code_clean") == 76, 1)).alias("func_code_76_count"),
    F.count(F.when(F.col("func_code_clean") == 79, 1)).alias("func_code_79_count"),
    F.count(F.when(F.col("func_code_clean") == 75, 1)).alias("func_code_75_count"),
    F.count(F.when(F.col("func_code_clean") == 80, 1)).alias("func_code_80_count"),
    F.count(F.when(F.col("func_code_clean") == 85, 1)).alias("func_code_85_count"),
    F.count(F.when(F.col("func_code_clean") == 4,  1)).alias("func_code_4_count"),
    F.count(F.when(F.col("func_code_clean").isin(5, 6, 15, 16), 1)).alias("write_func_codes_count"),
    F.max(F.col("has_corrupt_function_code")).alias("has_corrupt_function_code"),
    F.countDistinct("func_code_clean").alias("unique_function_codes"),

    # ── GRUPO 5: Request / Response
    F.count(F.when(
        ~F.col("desc_clean").rlike("(?i)response") &
         F.col("desc_clean").isNotNull(), 1)
    ).alias("read_requests"),
    F.count(F.when(
        F.col("desc_clean").rlike("(?i)response"), 1)
    ).alias("read_responses"),
    F.count(F.when(
        F.col("desc_clean").rlike("(?i)write|set.attribute"), 1)
    ).alias("write_operations"),
    F.countDistinct("Modbus_Transaction_ID").alias("unique_transaction_ids"),

    # ── GRUPO 6: Transaction ID stats
    F.mean("tx_id_numeric").alias("transaction_id_mean"),
    F.stddev("tx_id_numeric").alias("transaction_id_std"),
    (F.max("tx_id_numeric") - F.min("tx_id_numeric")).alias("transaction_id_range"),

    # ── GRUPO 7: Puertos
    F.max(F.col("has_corrupt_dst_port")).alias("has_corrupt_dst_port"),
    F.max(F.when(
        F.col("service_clean").isNotNull() &
        (F.col("service_clean") != 44818), 1).otherwise(0)
    ).alias("has_non_standard_dst_port"),
    F.max(F.when(
        F.col("s_port_clean").isNotNull() &
        (~F.col("s_port_clean").isin(NORMAL_SRC_PORTS)), 1).otherwise(0)
    ).alias("has_anomalous_src_port"),
    F.countDistinct("s_port_clean").alias("unique_src_ports"),

    # ── GRUPO 8: Payload
    F.count(F.when(F.col("Modbus_Value").contains("0x"), 1)).alias("response_payload_packets"),
    F.count(F.when(F.col("Modbus_Value") == "Number of Elements: 1", 1)).alias("empty_payload_packets"),
    F.mean("payload_bytes").alias("avg_payload_bytes"),
    F.max("payload_bytes").alias("max_payload_bytes"),
    F.min("payload_bytes").alias("min_payload_bytes"),

    # ── GRUPO 9: SCADA Tags pivot (conteo por tag)
    *[F.count(F.when(F.col("SCADA_Tag") == tag, 1)).alias(f"tag_{tag}")
      for tag in SCADA_TAGS],
    F.countDistinct("SCADA_Tag").alias("unique_scada_tags"),

    # ── GRUPO 10: Etiqueta
    F.max(F.when(F.col("Tag") != "Normal", 1).otherwise(0)).alias("label")
)

# ══════════════════════════════════════════════════════════════════════════════
# 5. FEATURES DERIVADAS
# ══════════════════════════════════════════════════════════════════════════════

features = features \
    .withColumn("inbound_outbound_ratio",
        F.col("inbound_packets") / (F.col("outbound_packets") + 1)
    ) \
    .withColumn("request_response_ratio",
        F.col("read_requests") / (F.col("read_responses") + 1)
    ) \
    .withColumn("write_read_ratio",
        F.col("write_operations") / (F.col("read_requests") + 1)
    ) \
    .withColumn("anomalous_appl_ratio",
        F.col("anomalous_appl_packets") / (F.col("total_packets") + 1)
    )



In [0]:
display(features.limit(10))

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. JOIN CON CONNECTION DURATION
# ══════════════════════════════════════════════════════════════════════════════

# Minimal fix: Join features with df_durations, keeping all left columns and only non-conflicting right columns
left_cols = features.columns
right_cols = [col for col in df_durations.columns if col not in left_cols or col == 'window_id']

features = features.join(df_durations.select(right_cols), on="window_id", how="left")




In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. DETECCIÓN DE GAPS TEMPORALES
# ══════════════════════════════════════════════════════════════════════════════

w_gap = Window.orderBy("window_id")

features = features \
    .withColumn("prev_window_id", F.lag("window_id").over(w_gap)) \
    .withColumn("gap_before_seconds",
        F.when(
            F.col("prev_window_id").isNotNull(),
            (F.col("window_id") - F.col("prev_window_id")) * WINDOW_SECONDS
        ).otherwise(0)
    ) \
    .withColumn("is_after_gap",
        F.when(F.col("gap_before_seconds") > GAP_THRESHOLD_S, 1).otherwise(0)
    ) \
    .withColumn("session_id",
        F.sum("is_after_gap").over(w_gap)
    ) \
    .drop("prev_window_id")

# ══════════════════════════════════════════════════════════════════════════════
# 8. RELLENAR NULOS — ausencia de tráfico = 0, no null
# ══════════════════════════════════════════════════════════════════════════════

# Columnas numéricas que pueden quedar null tras el join de durations
fill_zero_cols = [
    "avg_connection_duration_ms",
    "max_connection_duration_ms",
    "min_connection_duration_ms",
    "transaction_id_std",
    "transaction_id_range",
    "transaction_id_mean",
    "avg_payload_bytes",
    "max_payload_bytes",
    "min_payload_bytes",
]

features = features.fillna(0, subset=fill_zero_cols)

antes = features.count()
for col in fill_zero_cols:
    features = features.filter(F.col(col).isNotNull())
despues = features.count()
print(f"Filas eliminadas: {antes - despues:,}")



In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. GUARDAR EN DELTA
# ══════════════════════════════════════════════════════════════════════════════

features_to_save = features.coalesce(64)

features_to_save.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(FEATURES_PATH)

print("✅ Features guardadas en Delta Lake.")

# ── Verificación final
df_feat = spark.read.format("delta").load(FEATURES_PATH)

summary = df_feat.agg(
    F.count("*").alias("total_ventanas"),
    F.sum(F.when(F.col("label") == 1, 1).otherwise(0)).alias("ventanas_ataque"),
    F.countDistinct("session_id").alias("sesiones_distintas")
).collect()[0]

print(f"Total ventanas:        {summary['total_ventanas']:,}")
print(f"Total features:        {len(df_feat.columns)}")
print(f"Ventanas con ataque:   {summary['ventanas_ataque']:,}")
print(f"Sesiones distintas:    {summary['sesiones_distintas']:,}")

display(df_feat.orderBy("window_id").limit(10))

In [0]:
display(
    df_feat.groupBy("session_id")
    .agg(
        F.count("*").alias("ventanas"),
        F.min("window_start").alias("inicio"),
        F.max("window_end").alias("fin"),
        F.max("gap_before_seconds").alias("gap_maximo_seg")
    )
    .orderBy("session_id")
)

In [0]:
df_sesion1 = df_feat.filter(F.col("session_id") == 1)

# ── 1. ESTADÍSTICAS GENERALES DE LOS GAPS
display(
    df_sesion1.select("gap_before_seconds").summary(
        "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
    )
)

# ── 2. DISTRIBUCIÓN POR TAMAÑO DE GAP
display(
    df_sesion1.groupBy(
        F.when(F.col("gap_before_seconds") == 1,  "1 seg (consecutivo)")
         .when(F.col("gap_before_seconds") <= 5,  "2-5 seg")
         .when(F.col("gap_before_seconds") <= 60, "6-60 seg")
         .when(F.col("gap_before_seconds") <= 3600, "1-60 min")
         .otherwise("más de 1 hora")
         .alias("rango_gap")
    )
    .agg(F.count("*").alias("num_ventanas"))
    .orderBy("rango_gap")
)

# ── 3. LOS 20 GAPS MÁS GRANDES
display(
    df_sesion1.filter(F.col("gap_before_seconds") > 1)
    .select("window_start", "window_end", "gap_before_seconds")
    .orderBy("gap_before_seconds", ascending=False)
    .limit(20)
)